# Bayesian Linear Regression with Missing Responses on `kidiq`

This notebook demonstrates a Gibbs sampler for Bayesian linear regression with missing child IQ responses following the provided implementation plan.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import invgamma, multivariate_normal
from numpy.linalg import inv

sns.set(style='whitegrid', context='notebook')
np.random.seed(2024)

data_path = 'kidiq.dta'
if os.path.exists(data_path):
    df = pd.read_stata(data_path)
    source_note = 'Loaded kidiq.dta from local path.'
else:
    n = 434
    mom_hs = np.random.binomial(1, 0.7, size=n)
    mom_iq = np.random.normal(100, 15, size=n)
    mom_work = np.random.choice([2, 3, 4], size=n, p=[0.3, 0.4, 0.3])
    mom_age = np.random.normal(27, 4, size=n)
    noise = np.random.normal(0, 10, size=n)
    kid_score = 25 + 12 * mom_hs + 0.6 * mom_iq - 1.5 * (mom_work == 4) + noise
    df = pd.DataFrame({
        'kid_score': kid_score,
        'mom_hs': mom_hs.astype(float),
        'mom_iq': mom_iq,
        'mom_work': mom_work,
        'mom_age': mom_age
    })
    source_note = 'Synthetic data generated because kidiq.dta was not found.'

display(source_note)
display(df.head())
display(df.describe(include='all'))


In [ ]:
# Introduce missing responses if necessary
df_working = df.copy()
missing_rate = 0.2
observed_mask = df_working['kid_score'].notna()
current_missing = (~observed_mask).sum() / len(df_working)
if current_missing < missing_rate:
    missing_indices = df_working[observed_mask].sample(frac=missing_rate, random_state=2024).index
    df_working.loc[missing_indices, 'kid_score'] = np.nan

obs_idx = df_working['kid_score'].notna().values
mis_idx = ~obs_idx
missing_share = mis_idx.mean()

display(f'Missing share in kid_score: {missing_share:.1%}')
display(df_working.head())


In [ ]:
# Build design matrix with intercept
y = df_working['kid_score'].values
X = np.column_stack([np.ones(len(df_working)), df_working[['mom_hs', 'mom_iq', 'mom_work', 'mom_age']].values])
p = X.shape[1]

beta0 = np.zeros(p)
c = 1e6
V0 = np.eye(p) * c
a0 = 0.01
b0 = 0.01

# Initialize missing responses with mean of observed y
y_init = y.copy()
mean_y_obs = np.nanmean(y_init)
y_init[mis_idx] = mean_y_obs

# OLS initialization using observed pairs
ols_mask = obs_idx
X_obs = X[ols_mask]
y_obs = y[ols_mask]
beta_ols = inv(X_obs.T @ X_obs) @ (X_obs.T @ y_obs)
resid = y_obs - X_obs @ beta_ols
sigma2_ols = np.var(resid)

display({'initial_beta': beta_ols, 'initial_sigma2': sigma2_ols})


In [ ]:
def gibbs_sampler(X, y, obs_mask, beta0, V0, a0, b0, iterations=4000, burn_in=1000):
    n, p = X.shape
    y_current = y.copy()
    # Ensure no NaNs remain for computation
    if np.any(np.isnan(y_current)):
        raise ValueError('Input y should have missing values imputed before sampling.')

    beta_samples = np.zeros((iterations - burn_in, p))
    sigma2_samples = np.zeros(iterations - burn_in)

    beta = inv(X[obs_mask].T @ X[obs_mask]) @ (X[obs_mask].T @ y_current[obs_mask])
    sigma2 = np.var(y_current[obs_mask] - X[obs_mask] @ beta)

    for it in range(iterations):
        V0_inv = inv(V0)
        Vn = inv(V0_inv + (X.T @ X) / sigma2)
        beta_n = Vn @ (V0_inv @ beta0 + (X.T @ y_current) / sigma2)
        beta = multivariate_normal.rvs(mean=beta_n, cov=Vn)

        resid = y_current - X @ beta
        an = a0 + n / 2
        bn = b0 + 0.5 * resid.T @ resid
        sigma2 = invgamma.rvs(a=an, scale=bn)

        # Update missing y values
        mean_mis = X[~obs_mask] @ beta
        y_current[~obs_mask] = np.random.normal(mean_mis, np.sqrt(sigma2))

        if it >= burn_in:
            idx = it - burn_in
            beta_samples[idx] = beta
            sigma2_samples[idx] = sigma2

    return beta_samples, sigma2_samples


In [ ]:
# Run Gibbs sampler
iterations = 4000
burn_in = 1000
beta_samples, sigma2_samples = gibbs_sampler(
    X=X,
    y=y_init,
    obs_mask=obs_idx,
    beta0=beta0,
    V0=V0,
    a0=a0,
    b0=b0,
    iterations=iterations,
    burn_in=burn_in
)

beta_mean = beta_samples.mean(axis=0)
beta_ci = np.quantile(beta_samples, [0.025, 0.975], axis=0)
sigma2_mean = sigma2_samples.mean()
sigma2_ci = np.quantile(sigma2_samples, [0.025, 0.975])

display(pd.DataFrame({
    'parameter': ['Intercept', 'mom_hs', 'mom_iq', 'mom_work', 'mom_age'],
    'post_mean': beta_mean,
    'ci_lower': beta_ci[0],
    'ci_upper': beta_ci[1]
}))
display({'sigma2_mean': sigma2_mean, 'sigma2_95ci': sigma2_ci})


In [ ]:
# Complete-case OLS comparison
complete_df = df_working.dropna(subset=['kid_score'])
X_cc = np.column_stack([np.ones(len(complete_df)), complete_df[['mom_hs', 'mom_iq', 'mom_work', 'mom_age']].values])
y_cc = complete_df['kid_score'].values
beta_cc = inv(X_cc.T @ X_cc) @ (X_cc.T @ y_cc)
resid_cc = y_cc - X_cc @ beta_cc
sigma2_cc = np.var(resid_cc)

display(pd.Series(beta_cc, index=['Intercept', 'mom_hs', 'mom_iq', 'mom_work', 'mom_age'], name='OLS (complete cases)'))
display({'sigma2_ols_complete_cases': sigma2_cc})


In [ ]:
# Trace plots for beta coefficients
fig, axes = plt.subplots(beta_samples.shape[1], 1, figsize=(8, 10), sharex=True)
params = ['Intercept', 'mom_hs', 'mom_iq', 'mom_work', 'mom_age']
for i, ax in enumerate(axes):
    ax.plot(beta_samples[:, i], color='steelblue', alpha=0.7, linewidth=0.8)
    ax.set_ylabel(params[i])
axes[-1].set_xlabel('Iteration (post burn-in)')
fig.suptitle('Trace Plots for Beta Samples', fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
trace_path = 'beta_traces.png'
plt.savefig(trace_path, dpi=150)
plt.close(fig)
display(f'Trace plots saved to {trace_path}')


In [ ]:
# Posterior density plots
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.kdeplot(beta_samples[:, 1], ax=axes[0], color='darkgreen', fill=True)
axes[0].set_title('Posterior of mom_hs coefficient')
axes[0].axvline(0, color='black', linestyle='--', linewidth=1)
sns.kdeplot(sigma2_samples, ax=axes[1], color='purple', fill=True)
axes[1].set_title('Posterior of sigma^2')
for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.5)
fig.tight_layout()
density_path = 'posterior_densities.png'
plt.savefig(density_path, dpi=150)
plt.close(fig)
display(f'Density plots saved to {density_path}')
